# Reddit Parsing + Raw Schema

This notebook:

1. Parses saved Reddit `.json` files.
2. Builds post-, interaction-, and user-level staging tables.
3. Builds the cross-platform Reddit Raw Schema.
4. Validates the required Raw Schema fields.

## Privacy / repository rules

- No machine-specific paths are stored in this notebook.
- `PROJECT_AUTHOR_SALT` must be provided through the environment.
- Do **not** commit the actual SALT.
- Raw/interim outputs are written under `data/interim/`, which should remain gitignored.


In [ ]:
import json
import os
import re
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter

import numpy as np
import pandas as pd


In [ ]:
URL_PATTERN = re.compile(r"https?://\S+|www\.\S+", flags=re.IGNORECASE)
WORD_PATTERN = re.compile(r"\b\w+\b", flags=re.UNICODE)


def utc_datetime(timestamp):
    """Convert a Unix timestamp to a timezone-aware pandas Timestamp."""
    if timestamp is None:
        return pd.NaT
    return pd.to_datetime(timestamp, unit="s", utc=True)


def clean_text(text):
    """Normalize whitespace while retaining the original wording."""
    if not isinstance(text, str):
        return ""
    return re.sub(r"\s+", " ", text).strip()


def text_features(text):
    """Lightweight features useful for later bot/anomaly detection."""
    text = clean_text(text)
    words = WORD_PATTERN.findall(text.lower())
    urls = URL_PATTERN.findall(text)
    unique_words = set(words)

    return {
        "char_count": len(text),
        "word_count": len(words),
        "unique_word_count": len(unique_words),
        "lexical_diversity": len(unique_words) / len(words) if words else 0.0,
        "url_count": len(urls),
        "has_url": bool(urls),
        "newline_count": text.count("\n"),
    }


def parse_reddit_json(json_file_path):
    """
    Parse one Reddit post JSON endpoint response.

    Returns
    -------
    post_row : dict
    interaction_rows : list[dict]
        Contains both top-level comments and replies.
    """
    json_file_path = Path(json_file_path)
    with json_file_path.open("r", encoding="utf-8") as f:
        data = json.load(f)

    if not isinstance(data, list) or len(data) < 2:
        raise ValueError(f"Unexpected Reddit JSON structure: {json_file_path}")

    post_children = data[0].get("data", {}).get("children", [])
    if not post_children:
        raise ValueError(f"Post information is missing: {json_file_path}")

    post_data = post_children[0].get("data", {})
    post_id = post_data.get("id")
    post_created = utc_datetime(post_data.get("created_utc"))

    post_row = {
        "post_id": post_id,
        "title_raw": post_data.get("title") if isinstance(post_data.get("title"), str) else "",
        "selftext_raw": post_data.get("selftext") if isinstance(post_data.get("selftext"), str) else "",
        "post_fullname": post_data.get("name"),
        "subreddit": post_data.get("subreddit"),
        "title": clean_text(post_data.get("title")),
        "selftext": clean_text(post_data.get("selftext")),
        "author": post_data.get("author"),
        "author_fullname": post_data.get("author_fullname"),
        "score": post_data.get("score"),
        "upvote_ratio": post_data.get("upvote_ratio"),
        "num_comments_reported": post_data.get("num_comments"),
        "created_utc": post_created,
        "domain": post_data.get("domain"),
        "is_self": post_data.get("is_self"),
        "is_video": post_data.get("is_video"),
        "over_18": post_data.get("over_18"),
        "url": post_data.get("url"),
        "permalink": (
            f"https://www.reddit.com{post_data.get('permalink')}"
            if post_data.get("permalink") else None
        ),
        "source_file": json_file_path.name,
    }

    interactions = []

    def walk_comments(children, root_comment_id=None, ancestor_ids=None):
        ancestor_ids = ancestor_ids or []

        for child in children or []:
            kind = child.get("kind")

            # t1 = comment/reply. kind='more' is a placeholder and has no body.
            if kind != "t1":
                continue

            c = child.get("data", {})
            comment_id = c.get("id")
            parent_fullname = c.get("parent_id")
            is_top_level = isinstance(parent_fullname, str) and parent_fullname.startswith("t3_")
            parent_comment_id = (
                parent_fullname.removeprefix("t1_")
                if isinstance(parent_fullname, str) and parent_fullname.startswith("t1_")
                else None
            )
            this_root = comment_id if is_top_level else root_comment_id
            depth = c.get("depth")
            if depth is None:
                depth = len(ancestor_ids)

            replies_obj = c.get("replies")
            reply_children = []
            if isinstance(replies_obj, dict):
                reply_children = replies_obj.get("data", {}).get("children", []) or []

            direct_reply_ids = [
                r.get("data", {}).get("id")
                for r in reply_children
                if r.get("kind") == "t1"
            ]

            body_raw = c.get("body") if isinstance(c.get("body"), str) else ""
            body = clean_text(body_raw)
            created = utc_datetime(c.get("created_utc"))
            features = text_features(body)

            # Append parent BEFORE traversing replies.
            interactions.append({
                "post_id": post_id,
                "comment_id": comment_id,
                "comment_fullname": c.get("name"),
                "record_type": "comment" if is_top_level else "reply",
                "is_top_level": is_top_level,
                "parent_fullname": parent_fullname,
                "parent_comment_id": parent_comment_id,
                "root_comment_id": this_root,
                "depth": int(depth),
                "thread_path": "/".join([*ancestor_ids, comment_id]) if comment_id else None,
                "author": c.get("author"),
                "author_fullname": c.get("author_fullname"),
                "body_raw": body_raw,
                "body": body,
                "score": c.get("score"),
                "ups": c.get("ups"),
                "controversiality": c.get("controversiality"),
                "created_utc": created,
                "seconds_after_post": (
                    (created - post_created).total_seconds()
                    if pd.notna(created) and pd.notna(post_created) else np.nan
                ),
                "direct_reply_count": len(direct_reply_ids),
                "has_replies": len(direct_reply_ids) > 0,
                "is_submitter": c.get("is_submitter"),
                "is_edited": c.get("edited") not in (False, None),
                "distinguished": c.get("distinguished"),
                "stickied": c.get("stickied"),
                "collapsed": c.get("collapsed"),
                "locked": c.get("locked"),
                "permalink": (
                    f"https://www.reddit.com{c.get('permalink')}"
                    if c.get("permalink") else None
                ),
                "source_file": json_file_path.name,
                **features,
            })

            walk_comments(
                reply_children,
                root_comment_id=this_root,
                ancestor_ids=[*ancestor_ids, comment_id] if comment_id else ancestor_ids,
            )

    comment_children = data[1].get("data", {}).get("children", [])
    walk_comments(comment_children)

    return post_row, interactions


In [ ]:
def add_interaction_features(interactions_df):
    """Add parent-author and response-time features after all rows are parsed."""
    if interactions_df.empty:
        return interactions_df.copy()

    df = interactions_df.copy()
    parent_lookup = df.set_index("comment_id")[["author", "created_utc", "body"]].rename(
        columns={
            "author": "parent_author",
            "created_utc": "parent_created_utc",
            "body": "parent_body",
        }
    )

    df = df.join(parent_lookup, on="parent_comment_id")
    df["response_time_seconds"] = (
        df["created_utc"] - df["parent_created_utc"]
    ).dt.total_seconds()
    df["is_self_reply"] = (
        df["author"].notna()
        & df["parent_author"].notna()
        & df["author"].eq(df["parent_author"])
    )
    return df


def normalized_text(text):
    """Normalization used only for duplicate-text measurements."""
    text = clean_text(text).lower()
    text = URL_PATTERN.sub("<url>", text)
    text = re.sub(r"[^\w\s<>]", " ", text, flags=re.UNICODE)
    return re.sub(r"\s+", " ", text).strip()


def build_user_features(interactions_df):
    """Aggregate interaction-level data into one row per Reddit author."""
    if interactions_df.empty:
        return pd.DataFrame()

    df = interactions_df.copy()
    df = df[df["author"].notna() & ~df["author"].isin(["[deleted]", "AutoModerator"])]
    df["normalized_body"] = df["body"].map(normalized_text)
    df = df.sort_values(["author", "created_utc"])
    df["prev_user_time"] = df.groupby("author")["created_utc"].shift(1)
    df["interarrival_seconds"] = (
        df["created_utc"] - df["prev_user_time"]
    ).dt.total_seconds()
    df["hour_utc"] = df["created_utc"].dt.hour

    rows = []
    for author, g in df.groupby("author", sort=False):
        text_counts = g.loc[g["normalized_body"].ne(""), "normalized_body"].value_counts()
        duplicate_items = int(text_counts[text_counts > 1].sum()) if not text_counts.empty else 0
        active_hours = int(g["hour_utc"].nunique())

        rows.append({
            "author": author,
            "author_fullname": g["author_fullname"].dropna().iloc[0] if g["author_fullname"].notna().any() else None,
            "total_interactions": len(g),
            "posts_participated": g["post_id"].nunique(),
            "top_level_comments": int(g["is_top_level"].sum()),
            "replies": int((~g["is_top_level"]).sum()),
            "reply_ratio": float((~g["is_top_level"]).mean()),
            "unique_parent_authors": g["parent_author"].nunique(dropna=True),
            "unique_threads": g["root_comment_id"].nunique(dropna=True),
            "max_depth": int(g["depth"].max()),
            "avg_depth": float(g["depth"].mean()),
            "mean_score": float(g["score"].mean()),
            "median_score": float(g["score"].median()),
            "mean_word_count": float(g["word_count"].mean()),
            "median_word_count": float(g["word_count"].median()),
            "mean_lexical_diversity": float(g["lexical_diversity"].mean()),
            "url_interaction_ratio": float(g["has_url"].mean()),
            "total_urls": int(g["url_count"].sum()),
            "exact_duplicate_ratio": duplicate_items / len(g) if len(g) else 0.0,
            "unique_text_ratio": g["normalized_body"].nunique() / len(g) if len(g) else 0.0,
            "median_interarrival_seconds": float(g["interarrival_seconds"].median()),
            "min_interarrival_seconds": float(g["interarrival_seconds"].min()),
            "median_response_time_seconds": float(g["response_time_seconds"].median()),
            "very_fast_reply_ratio_60s": float(
                (g["response_time_seconds"].between(0, 60, inclusive="both")).mean()
            ),
            "active_utc_hours": active_hours,
            "hour_coverage_ratio": active_hours / 24,
            "self_reply_ratio": float(g["is_self_reply"].mean()),
            "first_activity_utc": g["created_utc"].min(),
            "last_activity_utc": g["created_utc"].max(),
        })

    return pd.DataFrame(rows)


In [ ]:
def parse_directory(directory):
    """Parse all .json files in a directory and return three DataFrames."""
    directory = Path(directory)
    json_files = sorted(directory.glob("*.json"))

    all_posts = []
    all_interactions = []
    errors = []

    for file_path in json_files:
        try:
            post, interactions = parse_reddit_json(file_path)
            all_posts.append(post)
            all_interactions.extend(interactions)
        except Exception as exc:
            errors.append({"source_file": file_path.name, "error": repr(exc)})

    posts_df = pd.DataFrame(all_posts)
    interactions_df = add_interaction_features(pd.DataFrame(all_interactions))
    users_df = build_user_features(interactions_df)
    errors_df = pd.DataFrame(errors)

    return posts_df, interactions_df, users_df, errors_df


In [ ]:
from pathlib import Path

def find_project_root() -> Path:
    """
    Find the repository root from the current working directory.
    Expected repo markers: src/ and README.md.
    """
    cwd = Path.cwd().resolve()

    for candidate in [cwd, *cwd.parents]:
        if (
            (candidate / "src").exists()
            and (candidate / "README.md").exists()
        ):
            return candidate

    # Fallback: current working directory.
    return cwd


PROJECT_ROOT = find_project_root()
print("PROJECT_ROOT:", PROJECT_ROOT)


In [ ]:
RAW_JSON_DIR = (
    PROJECT_ROOT
    / "data"
    / "interim"
    / "reddit"
    / "raw_json_audit"
    / "raw_reddit_json"
)

PARSED_OUTPUT_DIR = (
    PROJECT_ROOT
    / "data"
    / "interim"
    / "reddit"
    / "parsed"
)

PARSED_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

print("RAW_JSON_DIR:", RAW_JSON_DIR)
print("JSON files:", len(list(RAW_JSON_DIR.glob("*.json"))))

posts_df, interactions_df, users_df, errors_df = parse_directory(
    RAW_JSON_DIR
)

print("posts_df:", posts_df.shape)
print("interactions_df:", interactions_df.shape)
print("users_df:", users_df.shape)
print("errors_df:", errors_df.shape)

if posts_df.empty:
    raise RuntimeError(
        "No posts were parsed. Check RAW_JSON_DIR and parse_errors."
    )

posts_df.to_parquet(
    PARSED_OUTPUT_DIR / "posts.parquet",
    index=False,
)
interactions_df.to_parquet(
    PARSED_OUTPUT_DIR / "interactions.parquet",
    index=False,
)
users_df.to_parquet(
    PARSED_OUTPUT_DIR / "users.parquet",
    index=False,
)
errors_df.to_csv(
    PARSED_OUTPUT_DIR / "parse_errors.csv",
    index=False,
)

# CSV copies are convenient for inspection.
posts_df.to_csv(
    PARSED_OUTPUT_DIR / "posts.csv",
    index=False,
)
interactions_df.to_csv(
    PARSED_OUTPUT_DIR / "interactions.csv",
    index=False,
)
users_df.to_csv(
    PARSED_OUTPUT_DIR / "users.csv",
    index=False,
)

print("Saved parsed staging outputs to:", PARSED_OUTPUT_DIR)


## Author hashing

The Raw Schema requires a fixed project SALT.

Set `PROJECT_AUTHOR_SALT` **outside the notebook** before running this section.
Do not paste or commit the real secret into the repository.


In [ ]:
import os

if not os.environ.get("PROJECT_AUTHOR_SALT"):
    raise RuntimeError(
        "PROJECT_AUTHOR_SALT is not set. "
        "Set the fixed project secret in your environment before running "
        "the Raw Schema export."
    )

print("PROJECT_AUTHOR_SALT is available in the environment.")


In [ ]:
# %% [Raw Schema v0.2 export]
# Build one Reddit output table that follows raw_schema_v02.md.
#
# IMPORTANT:
# - This is the cross-platform RAW SCHEMA table for Reddit comments/replies.
# - No raw author username / author ID is written to this output.
# - text_raw uses body_raw, NOT the cleaned `body` field.
# - Existing feature-engineering columns such as char_count, depth, thread_path,
#   etc. stay in staging/interactions_df and are NOT included here.

import hashlib
import os
from pathlib import Path

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

PROJECT_START = pd.Timestamp("2026-02-28T00:00:00Z")
PROJECT_END = pd.Timestamp("2026-07-22T23:59:59Z")

QUERY_VERSION = "v3.0"
COLLECTOR_VERSION = "reddit_raw_json_parser_v0.2"

# These are the provenance files created during parent discovery / JSON fetch.
AUDIT_DIR = (
    PROJECT_ROOT
    / "data"
    / "interim"
    / "reddit"
    / "raw_json_audit"
)

MASTER_PARENT_FILE = (
    AUDIT_DIR
    / "master_parent_posts_dedup.csv"
)

FETCH_LOG_FILE = (
    AUDIT_DIR
    / "raw_json_fetch_log.csv"
)

RAW_SCHEMA_OUTPUT_DIR = (
    PROJECT_ROOT
    / "data"
    / "interim"
    / "reddit"
    / "raw_schema"
)

RAW_SCHEMA_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

RAW_SCHEMA_CSV = RAW_SCHEMA_OUTPUT_DIR / "reddit_raw_schema.csv"
RAW_SCHEMA_PARQUET = RAW_SCHEMA_OUTPUT_DIR / "reddit_raw_schema.parquet"


# ------------------------------------------------------------
# Required project SALT
# ------------------------------------------------------------

SALT = os.environ.get("PROJECT_AUTHOR_SALT")

if not SALT:
    raise RuntimeError(
        "PROJECT_AUTHOR_SALT is not set. "
        "Set one fixed secret SALT for the whole project before building author_hash."
    )


def sha256_text(value):
    if not isinstance(value, str) or not value:
        return ""
    return hashlib.sha256(value.encode("utf-8")).hexdigest()


def author_to_hash(
    author_id,
    platform_content_id,
):
    if (
        isinstance(author_id, str)
        and author_id
        and author_id not in {
            "[deleted]",
            "deleted",
        }
    ):
        value = (
            f"reddit:{author_id}:{SALT}"
        )

    else:
        value = (
            f"reddit:"
            f"unavailable_author:"
            f"{platform_content_id}:"
            f"{SALT}"
        )

    return hashlib.sha256(
        value.encode("utf-8")
    ).hexdigest()


def iso_z(value):
    """Return ISO-8601 UTC using Z, or empty string."""
    ts = pd.to_datetime(value, utc=True, errors="coerce")

    if pd.isna(ts):
        return ""

    return ts.strftime("%Y-%m-%dT%H:%M:%S.%fZ")


def project_week_fields(value):
    """
    Returns:
        project_week, in_window, is_partial_week
    """
    ts = pd.to_datetime(value, utc=True, errors="coerce")

    if pd.isna(ts):
        return "OUT", False, False

    if not (PROJECT_START <= ts <= PROJECT_END):
        return "OUT", False, False

    week_number = (ts - PROJECT_START).days // 7 + 1

    return (
        f"W{week_number:02d}",
        True,
        week_number == 21,
    )


def normalize_multi_ids(value):
    """
    Raw Schema requires ';' as the separator for matched_query_ids.
    Existing discovery files used ' | '.
    """
    if pd.isna(value) or value is None:
        return ""

    pieces = [
        item.strip()
        for item in str(value).replace("|", ";").split(";")
        if item.strip()
    ]

    return ";".join(sorted(set(pieces)))


def first_multi_id(value):
    normalized = normalize_multi_ids(value)

    if not normalized:
        return ""

    return normalized.split(";")[0]


# ------------------------------------------------------------
# Load provenance
# ------------------------------------------------------------

parent_meta = pd.read_csv(MASTER_PARENT_FILE)

fetch_log = pd.read_csv(FETCH_LOG_FILE)

# Keep the most recent successful JSON fetch for each post.
fetch_success = (
    fetch_log.loc[
        fetch_log["status"].eq("saved_raw_json")
    ]
    .copy()
)

fetch_success["finished_at_utc_dt"] = pd.to_datetime(
    fetch_success["finished_at_utc"],
    utc=True,
    errors="coerce",
)

fetch_success = (
    fetch_success
    .sort_values("finished_at_utc_dt")
    .drop_duplicates(
        subset=["post_id"],
        keep="last",
    )
)

parent_meta["post_id"] = parent_meta["post_id"].astype("string")
fetch_success["post_id"] = fetch_success["post_id"].astype("string")


# ------------------------------------------------------------
# Prepare lookup tables from parsed Reddit JSON
# ------------------------------------------------------------

posts_schema_lookup = posts_df.copy()

posts_schema_lookup["post_id"] = (
    posts_schema_lookup["post_id"]
    .astype("string")
)

post_keep = [
    "post_id",
    "title_raw",
    "title",
    "num_comments_reported",
]

post_keep = [
    col
    for col in post_keep
    if col in posts_schema_lookup.columns
]

posts_schema_lookup = (
    posts_schema_lookup[
        post_keep
    ]
    .drop_duplicates(
        subset=["post_id"],
        keep="first",
    )
)


# ------------------------------------------------------------
# Start from one row per comment/reply
# ------------------------------------------------------------

schema_source = interactions_df.copy()

schema_source["post_id"] = (
    schema_source["post_id"]
    .astype("string")
)

schema_source = schema_source.merge(
    posts_schema_lookup,
    on="post_id",
    how="left",
    suffixes=("", "_post"),
)

schema_source = schema_source.merge(
    parent_meta[
        [
            "post_id",
            "subreddit",
            "matched_query_ids",
            "matched_source_ids",
            "matched_search_terms",
            "discovery_routes",
        ]
    ],
    on="post_id",
    how="left",
    suffixes=("", "_provenance"),
)

schema_source = schema_source.merge(
    fetch_success[
        [
            "post_id",
            "finished_at_utc",
        ]
    ],
    on="post_id",
    how="left",
)


# ------------------------------------------------------------
# Per-post parsed counts
# ------------------------------------------------------------

items_kept_lookup = (
    schema_source
    .groupby("post_id", dropna=False)
    .size()
    .rename("items_kept")
)

schema_source = schema_source.join(
    items_kept_lookup,
    on="post_id",
)


# ------------------------------------------------------------
# Build Raw Schema rows
# ------------------------------------------------------------

rows = []

for row in schema_source.itertuples(index=False):

    created_at = getattr(row, "created_utc", pd.NaT)

    project_week, in_window, is_partial_week = (
        project_week_fields(created_at)
    )

    comment_fullname = getattr(
        row,
        "comment_fullname",
        "",
    ) or ""

    parent_fullname = getattr(
        row,
        "parent_fullname",
        "",
    ) or ""

    is_top_level = bool(
        getattr(
            row,
            "is_top_level",
            False,
        )
    )

    content_type = (
        "comment"
        if is_top_level
        else "reply"
    )

    # For top-level Reddit comments Raw Schema requires parent_id blank.
    parent_id = (
        ""
        if is_top_level
        else parent_fullname
    )

    matched_query_ids = normalize_multi_ids(
        getattr(
            row,
            "matched_query_ids",
            "",
        )
    )

    source_ids = normalize_multi_ids(
        getattr(
            row,
            "matched_source_ids",
            "",
        )
    )

    discovery_routes = normalize_multi_ids(
        getattr(
            row,
            "discovery_routes",
            "",
        )
    )

    collected_at = getattr(
        row,
        "finished_at_utc",
        "",
    )

    permalink = getattr(
        row,
        "permalink",
        "",
    ) or ""

    raw_body = getattr(
        row,
        "body_raw",
        "",
    )

    author = getattr(
        row,
        "author",
        "",
    )

    author_fullname = getattr(
        row,
        "author_fullname",
        "",
    )

    # Prefer platform author ID if available; fallback to author name only
    # in memory for hashing. Neither is written to the Raw Schema output.
    author_identity = (
        author_fullname
        if isinstance(author_fullname, str) and author_fullname
        else author
    )

    if raw_body == "[deleted]":
        content_status = "deleted"
    elif raw_body == "[removed]":
        content_status = "removed"
    else:
        content_status = "active"

    post_title_raw = getattr(
        row,
        "title_raw",
        "",
    )

    if not isinstance(post_title_raw, str):
        post_title_raw = ""

    subreddit = getattr(
        row,
        "subreddit",
        "",
    )

    # In case merge suffixes created a provenance subreddit field.
    if not isinstance(subreddit, str) or not subreddit:
        subreddit = getattr(
            row,
            "subreddit_provenance",
            "",
        )

    num_comments_reported = getattr(
        row,
        "num_comments_reported",
        np.nan,
    )

    items_kept = getattr(
        row,
        "items_kept",
        np.nan,
    )

    # Existing fetch log predates the formal collection_run_id contract.
    # We create a transparent deterministic surrogate for this legacy batch.
    # Future collectors should write the true run ID directly.
    collected_ts = pd.to_datetime(
        collected_at,
        utc=True,
        errors="coerce",
    )

    if pd.notna(collected_ts):
        collection_run_id = (
            "legacy_reddit_json_"
            + collected_ts.strftime("%Y%m%d")
        )
    else:
        collection_run_id = "legacy_reddit_json_unknown"

    rows.append(
        {
            # ------------------------------
            # Core / Required
            # ------------------------------
            "platform": "reddit",
            "platform_content_id": comment_fullname,
            "content_type": content_type,
            "created_at_utc": iso_z(created_at),
            "collected_at_utc": iso_z(collected_at),
            "text_raw": raw_body,
            "author_hash": author_to_hash(
                author_identity,
                comment_fullname,
            ),
            "project_week": project_week,
            "in_window": in_window,
            "is_partial_week": is_partial_week,
            "query_id": first_multi_id(matched_query_ids),
            "collection_run_id": collection_run_id,

            # ------------------------------
            # Source / Provenance
            # ------------------------------
            "source_id": first_multi_id(source_ids),
            "source_container": (
                f"r/{subreddit}"
                if isinstance(subreddit, str) and subreddit
                else ""
            ),
            "source_container_id": "",
            "source_parent_id": (
                f"t3_{getattr(row, 'post_id', '')}"
                if getattr(row, "post_id", "")
                else ""
            ),
            "source_parent_title": post_title_raw,
            "parent_id": parent_id,
            "query_version": QUERY_VERSION,
            "discovery_route": first_multi_id(discovery_routes),
            "matched_query_ids": matched_query_ids,
            "collector_version": COLLECTOR_VERSION,
            "permalink_hash": sha256_text(permalink),
            "source_total_available": num_comments_reported,
            "sampling_applied": False,
            "items_kept": items_kept,
            "random_seed": "",

            # ------------------------------
            # Engagement
            # ------------------------------
            "engagement_score": getattr(row, "score", np.nan),
            "engagement_replies": getattr(
                row,
                "direct_reply_count",
                np.nan,
            ),
            "engagement_shares": np.nan,
            "engagement_quotes": np.nan,
            "engagement_views": np.nan,
            "engagement_collected_at_utc": iso_z(collected_at),

            # ------------------------------
            # Author
            # ------------------------------
            "author_is_verified": np.nan,
            "author_follower_count": np.nan,
            "author_account_age_days": np.nan,
            "author_is_submitter": getattr(
                row,
                "is_submitter",
                np.nan,
            ),
            "automation_risk_score": np.nan,

            # ------------------------------
            # Language / Status
            # ------------------------------
            "language_reported": "",
            "language_detected": "",
            "language_confidence": np.nan,
            "content_status": content_status,

            # ------------------------------
            # Geography
            # ------------------------------
            "geo_method": "",
            "country_or_region": "",
            "geo_confidence": "",
            "geo_granularity": "",
            "geo_limitations": "",
        }
    )


reddit_raw_schema_df = pd.DataFrame(rows)


# ------------------------------------------------------------
# Enforce exact cross-platform column order
# ------------------------------------------------------------

RAW_SCHEMA_COLUMNS = [
    "platform",
    "platform_content_id",
    "content_type",
    "created_at_utc",
    "collected_at_utc",
    "text_raw",
    "author_hash",
    "project_week",
    "in_window",
    "is_partial_week",
    "query_id",
    "collection_run_id",

    "source_id",
    "source_container",
    "source_container_id",
    "source_parent_id",
    "source_parent_title",
    "parent_id",
    "query_version",
    "discovery_route",
    "matched_query_ids",
    "collector_version",
    "permalink_hash",
    "source_total_available",
    "sampling_applied",
    "items_kept",
    "random_seed",

    "engagement_score",
    "engagement_replies",
    "engagement_shares",
    "engagement_quotes",
    "engagement_views",
    "engagement_collected_at_utc",

    "author_is_verified",
    "author_follower_count",
    "author_account_age_days",
    "author_is_submitter",
    "automation_risk_score",

    "language_reported",
    "language_detected",
    "language_confidence",
    "content_status",

    "geo_method",
    "country_or_region",
    "geo_confidence",
    "geo_granularity",
    "geo_limitations",
]

reddit_raw_schema_df = reddit_raw_schema_df.reindex(
    columns=RAW_SCHEMA_COLUMNS
)


# ------------------------------------------------------------
# Required-column validation
# ------------------------------------------------------------

REQUIRED_COLUMNS = [
    "platform",
    "platform_content_id",
    "content_type",
    "created_at_utc",
    "collected_at_utc",
    "text_raw",
    "author_hash",
    "project_week",
    "in_window",
    "is_partial_week",
    "query_id",
    "collection_run_id",
]

missing_required_columns = [
    column
    for column in REQUIRED_COLUMNS
    if column not in reddit_raw_schema_df.columns
]

if missing_required_columns:
    raise ValueError(
        "Missing Required Raw Schema columns: "
        f"{missing_required_columns}"
    )


# Duplicate check using the Raw Schema primary content identifier.
duplicate_content_ids = (
    reddit_raw_schema_df[
        "platform_content_id"
    ]
    .duplicated()
    .sum()
)


# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

reddit_raw_schema_df.to_csv(
    RAW_SCHEMA_CSV,
    index=False,
    encoding="utf-8-sig",
)

reddit_raw_schema_df.to_parquet(
    RAW_SCHEMA_PARQUET,
    index=False,
)


# ------------------------------------------------------------
# Quick contract report
# ------------------------------------------------------------

print("=== Reddit Raw Schema v0.2 ===")
print(f"Rows: {len(reddit_raw_schema_df):,}")
print(f"Columns: {len(reddit_raw_schema_df.columns)}")
print(f"Duplicate platform_content_id: {duplicate_content_ids:,}")
print(
    "Rows inside project window:",
    f"{reddit_raw_schema_df['in_window'].sum():,}",
)

print("\nContent type:")
print(
    reddit_raw_schema_df[
        "content_type"
    ].value_counts(
        dropna=False
    )
)

print("\nProject week:")
print(
    reddit_raw_schema_df[
        "project_week"
    ].value_counts()
    .sort_index()
)

print("\nMissing values in Required columns:")
print(
    reddit_raw_schema_df[
        REQUIRED_COLUMNS
    ]
    .replace("", np.nan)
    .isna()
    .sum()
)

print(f"\nSaved CSV: {RAW_SCHEMA_CSV}")
print(f"Saved Parquet: {RAW_SCHEMA_PARQUET}")

reddit_raw_schema_df.head()


In [ ]:
REQUIRED_COLUMNS = [
    "platform",
    "platform_content_id",
    "content_type",
    "created_at_utc",
    "collected_at_utc",
    "text_raw",
    "author_hash",
    "project_week",
    "in_window",
    "is_partial_week",
    "query_id",
    "collection_run_id",
]

reddit_raw_schema_df[
    REQUIRED_COLUMNS
].replace("", np.nan).isna().sum()

In [ ]:
reddit_raw_schema_df[reddit_raw_schema_df['in_window'] == True].shape[0]